In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

## Άσκηση - Δειγματοληψία και aliasing στο χρόνο και στη συχνότητα

Στο Κεφάλαιο αυτό συζητήσαμε για το θεώρημα των Shannon-Nyquist και τον τρόπο με τον οποίο - υπακούοντας στο θεώρημα αυτό - μπορούμε να ανακατασκευάσουμε ένα σήμα συνεχούς χρόνου πλήρως και ακριβώς από τα δείγματά του. Για να συμβεί αυτό απαιτούνται δυο προϋποθέσεις:

- Η δειγματοληψία να είναι **ομοιόμορφη** (δηλ. παίρνουμε δείγματα από το σήμα συνεχούς χρόνου σε σταθερά χρονικά διαστήματα, όπως αυτά ορίζονται από την **περίοδο δειγματοληψίας** $T_s$)
- Το σήμα συνεχούς χρόνου είναι ένα σήμα **βασικής ζώνης** (δηλ. όλο του το φασματικό περιεχόμενο βρίσκεται συγκεντρωμένο γύρω από το 0: $|X(f)| = 0, \: |f| > B$, με $B$ τη μέγιστη μη μηδενικού πλάτους συχνότητα του σήματος)

Αν ισχύουν τα παραπάνω, μπορούμε να επιλέξουμε μια **συχνότητα δειγματοληψίας** $f_s = 1/T_s$, τέτοια ώστε

$$f_s > 2B = 2f_{max}$$

Έτσι, τα δείγματα που θα πάρουμε είναι ικανά να μας δώσουν πίσω το σήμα συνεχούς χρόνου με χρήση της συνάρτησης παρεμβολής $\texttt{sinc}()$, όπως είδαμε σε αυτό το Κεφάλαιο:

$$x(t) = \sum_{n=-\infty}^{+\infty}x(nT_s)\mathrm{sinc}\left(\frac{t-nT_s}{T_s}\right)$$

Σε αυτήν την άσκηση, θα δούμε την πράξη της δειγματοληψίας τόσο στο χρόνο όσο και στη συχνότητα, και θα παρατηρήσουμε το φαινόμενο του aliasing!

Ας δημιουργήσουμε ένα απλό ημίτονο συνεχούς χρόνου:

In [2]:
# Δημιουργία σήματος
def generate_signal(freq, duration=0.01, fs_high=100000):
    t = np.linspace(0, duration, int(fs_high * duration), endpoint=False)
    x = np.sin(2 * np.pi * freq * t)
    
    return t, x

Ας υλοποιήσουμε την ανακατασκευή του σήματος συνεχούς χρόνου από τα δείγματά του, όπως υπαγορεύεται από την παραπάνω εξίσωση (με έναν λίγο πιο έξυπνο τρόπο από το να γράψουμε ένα βρόχο επανάληψης για κάθε δείγμα).

In [3]:
# Συνάρτηση για ανακατασκευή σήματος με χρήση συνάρτησης παρεμβολής sinc
def sinc_reconstruct(xn, tn, t_recon):
    T = tn[1] - tn[0]                                           # Περίοδος δειγματοληψίας
    sinc_matrix = np.sinc((t_recon[:, None] - tn[None, :]) / T) # Πίνακς παρεμβολής sinc
    x = np.dot(sinc_matrix, xn)                                 # Ανακατασκευή σήματος

    return x

Στο παρακάτω widget μπορείτε να μετακινήσετε τους sliders για να ορίσετε τη συχνότητα του ημιτόνου συνεχούς χρόνου και τη συχνότητα δειγματοληψίας.

Δείτε τι συμβαίνει όταν:

- Επιλέξετε συχνότητα δειγματοληψίας μικρότερη από τη συχνότητα του ημιτόνου.
- Επιλέξετε συχνότητα δειγματοληψίας ίση με τη συχνότητα του ημιτόνου.
- Επιλέξετε συχνότητα δειγματοληψίας μεγαλύτερη από τη συχνότητα του ημιτόνου.

In [ ]:
@widgets.interact(freq=widgets.FloatSlider(value=500, min=100, max=2000, step=100),
                  fs=widgets.FloatSlider(value=4000, min=500, max=8000, step=500))
def interactive_sampling(freq, fs):
    t_cont, x_cont = generate_signal(freq)
    t_s = np.arange(0, 0.01, 1/fs)
    x_s = np.sin(2 * np.pi * freq * t_s)
    x_recon = sinc_reconstruct(x_s, t_s, t_cont)

    plt.figure(figsize=(10, 4))
    plt.plot(t_cont, x_cont, label='Σήμα συνεχούς χρόνου', linewidth=2)
    plt.stem(t_s, x_s, linefmt='r-', markerfmt='ro', basefmt=' ', label='Δείγματα')
    plt.plot(t_cont, x_recon, '--', label='Ανακατασκευή Sinc')
    plt.title(f'Δειγματοληψία με {int(fs)} Hz (Συχνότητα σήματος: {freq} Hz)')
    if fs < 2 * freq:
        plt.text(0.0, -1.5, 'Προκύπτει aliasing!', color='red')
    if fs == 2 * freq:
        plt.text(0.0, -1.5, 'Συχνότητα Nyquist!', color='green')
    plt.legend(loc='upper right')
    plt.ylim(-2, 2)
    plt.grid(True)
    plt.show()

interactive(children=(FloatSlider(value=500.0, description='freq', max=2000.0, min=100.0, step=100.0), FloatSl…

Γράψτε τις παρατηρήσεις σας για καθεμιά από τις παραπάνω περιπτώσεις στο παρακάτω κελί:

### Απάντηση:

- Έχουμε υποδειγματοληψία, και τότε παρατηρούμε το φαινόμενο του aliasing: το ανακατασκευασμένο σήμα είναι διαφορετικό από το αρχικό.
- Έχουμε οριακή δειγματοληψία, και τότε παρατηρούμε ότι το ανακατασκευασμένο σήμα είναι μηδενικό!
- Έχουμε υπερδειγματοληψία, το θεώρημα των Shannon-Nyquist ικανοποιείται και το ανακατασκευασμένο σήμα είναι ίδιο με το αρχικό (με εξαίρεση στα άκρα, που όμως είναι ζήτημα υπολογιστικό και όχι θεωρητικό)

---

Ας επαναλάβουμε την ίδια διαδικασία, όμως δείχνοντας τώρα την ίδια εικόνα στο χώρο της συχνότητας!

In [ ]:
def aliased_frequency(freq, fs):
    """Αναδίπλωση στο [0, fs/2)."""
    return abs(((freq + fs / 2) % fs) - fs / 2)


def draw_spectrum_impulses(freqs, height, color, label, linewidth=1.8):
    """Σχεδιάζουμε τις φασματικές γραμμές."""
    freqs = np.atleast_1d(freqs)
    if len(freqs) == 0:
        return
    for f in freqs:
        plt.annotate('', xy=(f, height), xytext=(f, 0), arrowprops=dict(arrowstyle='-|>', color=color,
                                     linewidth=linewidth, shrinkA=0, shrinkB=0, mutation_scale=13))
        
    plt.plot([], [], color=color, marker='^', linestyle='-', linewidth=linewidth, label=label)


@widgets.interact(freq=widgets.FloatSlider(value=500, min=100, max=2000, step=100),
                  fs=widgets.FloatSlider(value=4000, min=500, max=8000, step=500))
def interactive_spectrum(freq, fs):
    nyquist = fs / 2
    f_alias = aliased_frequency(freq, fs)
    has_alias = fs < 2 * freq
    view_limit = max(2.5 * fs, 2.5 * freq, 3000)
    k_max = int(np.ceil((view_limit + freq) / fs))

    replica_freqs = []
    for k in range(-k_max, k_max + 1):
        if k == 0:
            continue
        replica_freqs.extend([freq + k * fs, -freq + k * fs])
    replica_freqs = np.array([f for f in replica_freqs if -view_limit <= f <= view_limit])

    original_lines = np.array([-freq, freq])
    original_height = 0.5
    replica_height = 0.5
    alias_height = replica_height

    plt.figure(figsize=(10, 4))
    plt.axvspan(-nyquist, nyquist, color='tab:green', alpha=0.08, label='Διάστημα (-fs/2, fs/2)')
    plt.axvline(-nyquist, color='tab:green', linestyle='--', linewidth=1)
    plt.axvline(nyquist, color='tab:green', linestyle='--', linewidth=1)

    draw_spectrum_impulses(replica_freqs, replica_height, 'C1', 'Ρέπλικες λόγω δειγματοληψίας')
    draw_spectrum_impulses(original_lines, original_height, 'C0', 'Αρχικό σήμα')
    if has_alias:
        alias_lines = np.array([0]) if np.isclose(f_alias, 0) else np.array([-f_alias, f_alias])
        draw_spectrum_impulses(alias_lines, alias_height, 'r', 'Παρατηρούμενο aliasing')

    plt.title(f'Φάσμα με f0= {freq:.0f} Hz και fs = {fs:.0f} Hz')
    plt.xlabel('Συχνότητα (Hz)')
    plt.ylabel('Πλάτος (κανονικοποιημένο)')
    plt.xlim(-view_limit, view_limit)
    plt.ylim(0, 1.2)
    plt.grid(True, axis='x', alpha=0.35)
    plt.legend(loc='upper right')

    if has_alias:
        note = f'Ψευδής συχνότητα μέσα στο διάστημα \n(-fs/2, fs/2): {f_alias:.0f} Hz  (aliasing)'
    elif np.isclose(fs, 2 * freq):
        note = f'Ρυθμός Nyquist: f0 = fs/2 = {freq:.0f} Hz'
    else:
        note = 'Δεν υπάρχει aliasing: \nοι ρέπλικες είναι εκτός του (-fs/2, fs/2)'
    plt.text(-0.98 * view_limit, 1.08, note, color='red' if has_alias else 'black')
    plt.show()

interactive(children=(FloatSlider(value=500.0, description='freq', max=2000.0, min=100.0, step=100.0), FloatSl…

Ξανά, δείτε τι συμβαίνει όταν:

- Επιλέξετε συχνότητα δειγματοληψίας μικρότερη από τη συχνότητα του ημιτόνου.
- Επιλέξετε συχνότητα δειγματοληψίας ίση με τη συχνότητα του ημιτόνου.
- Επιλέξετε συχνότητα δειγματοληψίας μεγαλύτερη από τη συχνότητα του ημιτόνου.

Γράψτε τις παρατηρήσεις σας για καθεμιά από τις παραπάνω περιπτώσεις στο παρακάτω κελί:

### Απάντηση:

- Έχουμε υποδειγματοληψία, και τότε παρατηρούμε το φαινόμενο του aliasing: εμφανίζονται μέσα στο διάστημα $(-f_s/2, f_s/2)$ συχνότητες που δεν ανήκουν στο αρχικό φάσμα (ψευδώνυμες συχνότητες).
- Έχουμε οριακή δειγματοληψία, και τότε παρατηρούμε ότι το ανακατασκευασμένο σήμα είναι μηδενικό, γιατί το φίλτρο κόβει όλες τις συχνότητες!
- Έχουμε υπερδειγματοληψία, το θεώρημα των Shannon-Nyquist ικανοποιείται και το διάστημα $(-f_s/2, f_s/2)$ περιλαμβάνει μόνο συχνοτικό περιεχόμενο που ανήκει και στο αρχικό φάσμα.

---
---